In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a GPU program that implements top-p (nucleus) sampling for LLM inference.
</p>

<p>
  Top-p sampling is a text generation technique where you sample from the smallest set of tokens whose cumulative probability exceeds threshold p.
  This balances randomness and quality better than pure top-k or greedy sampling.
</p>

<p>
  Given logits (unnormalized scores) from a language model:
  <ol>
    <li>Convert logits to probabilities using softmax</li>
    <li>Sort tokens by probability (descending)</li>
    <li>Find the smallest set where cumulative probability ≥ p (the "nucleus")</li>
    <li>Renormalize the nucleus probabilities to sum to 1</li>
    <li>Sample a token from the nucleus using the provided random seed</li>
  </ol>
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>Ensure numerical stability when computing softmax</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
  logits = [1.0, 2.0, 3.0, 0.5]
  p = 0.9
  seed = 42

Output:
  sampled_token = 2 or 1
  (tokens with highest probabilities, sampled randomly)
</pre>

<h2>Example 2:</h2>
<pre>
Input:
  logits = [10.0, 1.0, 1.0]
  p = 0.5
  seed = 123

Output:
  sampled_token = 0
  (single token dominates the probability mass)
</pre>

<h2>Constraints</h2>
<ul>
  <li>3 &le; <code>vocab_size</code> &le; 50,000</li>
  <li>-100.0 &le; <code>logits[i]</code> &le; 100.0</li>
  <li>0.0 &lt; <code>p</code> &le; 1.0</li>
  <li>0 &le; <code>sampled_token</code> &lt; vocab_size</li>

  <li>Performance is measured with <code>vocab_size</code> = 50,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

extern "C" void solve(const float* logits, const float* p, const int* seed, int* sampled_token,
                      int vocab_size) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


@cute.jit
def solve(
    logits: cute.Tensor,
    p: cute.Tensor,
    seed: cute.Tensor,
    sampled_token: cute.Tensor,
    vocab_size: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


@jax.jit
def solve(logits: jax.Array, p: jax.Array, seed: jax.Array, vocab_size: int) -> jax.Array:
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.memory import UnsafePointer
from std.gpu import block_dim, block_idx, thread_idx


@export
def solve(
    logits: UnsafePointer[Float32, MutExternalOrigin],
    p: UnsafePointer[Float32, MutExternalOrigin],
    seed: UnsafePointer[Int32, MutExternalOrigin],
    sampled_token: UnsafePointer[Int32, MutExternalOrigin],
    vocab_size: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


def solve(
    logits: torch.Tensor,
    p: torch.Tensor,
    seed: torch.Tensor,
    sampled_token: torch.Tensor,
    vocab_size: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


def solve(
    logits: torch.Tensor,
    p: torch.Tensor,
    seed: torch.Tensor,
    sampled_token: torch.Tensor,
    vocab_size: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Top-p Sampling", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        logits: torch.Tensor,
        p: torch.Tensor,
        seed: torch.Tensor,
        sampled_token: torch.Tensor,
        vocab_size: int,
    ):
        assert logits.shape == (vocab_size,)
        assert p.shape == (1,)
        assert seed.shape == (1,)
        assert sampled_token.shape == (1,)
        assert logits.dtype == torch.float32
        assert p.dtype == torch.float32

        p_value = p.item()
        seed_value = seed.item()

        max_logit = torch.max(logits)
        exp_logits = torch.exp(logits - max_logit)
        probs = exp_logits / torch.sum(exp_logits)

        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(sorted_probs, dim=0)

        cutoff_idx = torch.searchsorted(cumsum, p_value, right=False).item()
        cutoff_idx = min(cutoff_idx + 1, vocab_size)

        nucleus_probs = sorted_probs[:cutoff_idx]
        nucleus_indices = sorted_indices[:cutoff_idx]

        nucleus_probs = nucleus_probs / torch.sum(nucleus_probs)

        torch.manual_seed(seed_value)
        sampled_idx = torch.multinomial(nucleus_probs, 1).item()
        sampled_token[0] = nucleus_indices[sampled_idx]

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "logits": (ctypes.POINTER(ctypes.c_float), "in"),
            "p": (ctypes.POINTER(ctypes.c_float), "in"),
            "seed": (ctypes.POINTER(ctypes.c_int32), "in"),
            "sampled_token": (ctypes.POINTER(ctypes.c_int32), "out"),
            "vocab_size": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        logits = torch.tensor([1.0, 2.0, 3.0, 0.5], device="cuda", dtype=torch.float32)
        p = torch.tensor([0.9], device="cuda", dtype=torch.float32)
        seed = torch.tensor([42], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)

        return {
            "logits": logits,
            "p": p,
            "seed": seed,
            "sampled_token": sampled_token,
            "vocab_size": 4,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        tests = []

        logits = torch.tensor([1.0, 2.0, 3.0], device="cuda", dtype=torch.float32)
        p = torch.tensor([0.95], device="cuda", dtype=torch.float32)
        seed = torch.tensor([123], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 3,
            }
        )

        logits = torch.randn(10, device="cuda", dtype=torch.float32)
        p = torch.tensor([0.9], device="cuda", dtype=torch.float32)
        seed = torch.tensor([456], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 10,
            }
        )

        logits = torch.randn(100, device="cuda", dtype=torch.float32) * 5.0
        p = torch.tensor([0.85], device="cuda", dtype=torch.float32)
        seed = torch.tensor([789], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 100,
            }
        )

        logits = torch.zeros(50, device="cuda", dtype=torch.float32)
        logits[0] = 10.0
        p = torch.tensor([0.5], device="cuda", dtype=torch.float32)
        seed = torch.tensor([111], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 50,
            }
        )

        logits = torch.randn(500, device="cuda", dtype=torch.float32) * 3.0
        p = torch.tensor([0.92], device="cuda", dtype=torch.float32)
        seed = torch.tensor([222], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 500,
            }
        )

        logits = torch.linspace(-5, 5, 200, device="cuda", dtype=torch.float32)
        p = torch.tensor([0.8], device="cuda", dtype=torch.float32)
        seed = torch.tensor([333], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 200,
            }
        )

        logits = torch.randn(1000, device="cuda", dtype=torch.float32) * 2.0
        p = torch.tensor([0.95], device="cuda", dtype=torch.float32)
        seed = torch.tensor([444], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 1000,
            }
        )

        logits = torch.randn(5000, device="cuda", dtype=torch.float32)
        p = torch.tensor([0.9], device="cuda", dtype=torch.float32)
        seed = torch.tensor([555], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)
        tests.append(
            {
                "logits": logits,
                "p": p,
                "seed": seed,
                "sampled_token": sampled_token,
                "vocab_size": 5000,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        vocab_size = 50000
        logits = torch.randn(vocab_size, device="cuda", dtype=torch.float32) * 3.0
        p = torch.tensor([0.9], device="cuda", dtype=torch.float32)
        seed = torch.tensor([999], device="cuda", dtype=torch.int32)
        sampled_token = torch.zeros(1, device="cuda", dtype=torch.int32)

        return {
            "logits": logits,
            "p": p,
            "seed": seed,
            "sampled_token": sampled_token,
            "vocab_size": vocab_size,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
